In [2]:
# Import essential libraries for authentication
# Spotipy is the go-to Python wrapper for the Spotify API
# Spotify uses OAuth 2.0, so we will import that
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Used to hide credentials
from dotenv import load_dotenv
import os

In [2]:
# Authentication via spotipy.Spotify()
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.getenv("SPOTIFY_CLIENT_ID"),
    client_secret=os.getenv("SPOTIFY_CLIENT_SECRET"),
    redirect_uri="http://127.0.0.1:8888/callback",
    scope="user-library-read user-top-read user-read-recently-played"
))

In [4]:
# Pull and store music data from multiple sources
# i.e. Liked Songs, Top Tracks/Artists, and Recently Played"
# Starting with Liked Songs
def get_liked_songs():
    results = sp.current_user_saved_tracks(limit=50),
    tracks = []
    while results:
        for item in results['items']:
            track = item['track']
            tracks.append({
                'id': track['id'],
                'name': track['name'],
                'artist': track['artists'][0]['name'],
                'artist_id': track['artists'][0]['id'],
                'popularity': track['popularity'],
                'source': 'liked'
        })
        results = sp.next(results)  # handles pagination
    return tracks

In [5]:
# Get top tracks 
# time_range options:
# - 'short_term'  = last 4 weeks
# - 'medium_term' = last 6 months (default)
# - 'long_term'   = all time
def get_top_tracks(time_range='medium_term', limit=50):
    results = sp.current_user_top_tracks(
        time_range=time_range,
        limit=limit
    )
    tracks = []
    for item in results['items']:
        tracks.append({
            'id': item['id'],
            'name': item['name'],
            'artist': item['artists'][0]['name'],
            'artist_id': item['artists'][0]['id'],
            'popularity': item['popularity'],
            'source': f'top_tracks_{time_range}'
        })
    return tracks

In [6]:
# Now get top artists
# Useful for seeding recommendations
# and filtering candidates by genre
def get_top_artists(time_range='medium_term', limit=50):
    results = sp.current_user_top_artists(
        time_range=time_range,
        limit=limit
    )
    artists = []
    for item in results['items']:
        artists.append({
            'id': item['id'],
            'name': item['name'],
            'genres': item['genres'],
            'popularity': item['popularity'],
            'source': f'top_artists_{time_range}'
        })
    return artists

In [7]:
# Lastly, get recently played
def get_recently_played(limit=50):
    results = sp.current_user_recently_played(limit=limit)
    tracks = []
    for item in results['items']:
        track = item['track']
        tracks.append({
            'id': track['id'],
            'name': track['name'],
            'artist': track['artists'][0]['name'],
            'artist_id': track['artists'][0]['id'],
            'played_at': item['played_at'],   # timestamp — useful for weighting
            'source': 'recently_played'
        })
    return tracks

In [8]:
# Now, need to fetch audio features for a list of tracks
# Spotify allows max 100 track IDs per request
def get_audio_features(tracks):
    track_ids = [t['id'] for t in tracks]
    features = []
    # batch into chunks of 100
    for i in range(0, len(track_ids), 100):
        batch = track_ids[i:i+100]
        results = sp.audio_features(batch)
        features.extend([f for f in results if f is not None])
    return features